# Code mẫu — Thi thử

Một buổi thi thử phải đi trọn đường: train, lưu model, dự đoán, ghi file, rồi chấm. Sau đó phải kiểm xem chỗ nào trên đường ấy có thể hỏng. Hai ví dụ nhỏ dưới đây, chạy trên CPU, tập trung vào đúng quy trình này.

## Ví dụ 1: Bộ chấm ghép nhãn theo ID, không theo vị trí tình cờ

Bộ chấm giả lập dùng log-loss cho ba ID. File hợp lệ phải có đúng thứ tự `001, 007, 010`. Nhãn thật do bộ chấm giữ riêng. Ta thử ba trường hợp: file đúng, file có cùng ba xác suất nhưng gán nhầm ID, và file thừa một cột index. File thừa cột phải bị từ chối trước khi chấm.

In [ ]:
from io import StringIO
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

truth = pd.Series([0, 1, 1], index=["001", "007", "010"])
def score_csv(text):
    frame = pd.read_csv(StringIO(text), dtype={"id": str})
    if frame.columns.tolist() != ["id", "probability"]:
        raise ValueError("schema phải là id,probability")
    if frame["id"].tolist() != truth.index.tolist() or not frame["id"].is_unique:
        raise ValueError("ID hoặc thứ tự không đúng")
    p = frame["probability"].to_numpy(dtype=float)
    if not np.isfinite(p).all() or not ((0 <= p) & (p <= 1)).all():
        raise ValueError("xác suất không hợp lệ")
    return log_loss(truth.to_numpy(), np.column_stack([1 - p, p]), labels=[0, 1])

good = pd.DataFrame({"id": truth.index, "probability": [0.1, 0.8, 0.9]})
swapped = good.copy()
swapped["probability"] = [0.9, 0.8, 0.1]
for name, frame in [("đúng mapping", good), ("đảo hai xác suất", swapped)]:
    text = frame.to_csv(index=False, lineterminator="\n")
    print(name, "log-loss:", f"{score_csv(text):.4f}")
try:
    score_csv(good.to_csv(index=True))
except ValueError as error:
    print("file thừa index:", error)
else:
    raise AssertionError("Scorer phải từ chối cột ngoài schema")
assert score_csv(good.to_csv(index=False)) < score_csv(swapped.to_csv(index=False))

đúng mapping log-loss: 0.1446
đảo hai xác suất log-loss: 1.6094
file thừa index: schema phải là id,probability


File đúng có log-loss 0.1446. Đảo xác suất của mẫu đầu và mẫu cuối, mọi giá trị vẫn nằm trong khoảng 0 tới 1, nhưng hai mẫu trở thành dự đoán sai mà rất tự tin, và log-loss vọt lên 1.6094. Quên `index=False` khi ghi CSV thì pandas thêm một cột index, và file bị từ chối hẳn.

Đây là hai tầng lỗi khác nhau. Sai định dạng thì file không chấm được. Đúng định dạng mà gán nhầm ID thì file chấm được, nhưng điểm rất tệ. Bộ chấm ở đây có nhãn vì đây là bài tập tại chỗ. Ở ví dụ sau, phần code dự đoán tuyệt đối không được chạm vào nhãn.

## Ví dụ 2: Train, lưu, nạp lại, và chấm một file thật

Ta sinh 120 mẫu có hai đặc trưng, với seed cố định, và giữ 30 mẫu làm validation. Một Pipeline gồm ba bước: điền chỗ thiếu bằng median, chuẩn hóa, và logistic regression. Model được lưu bằng pickle, và chỉ nạp lại chính file vừa tạo, trong cùng môi trường. Đừng bao giờ dùng pickle để mở file không rõ nguồn, vì nó có thể chạy mã tùy ý.

In [ ]:
import json
import pickle
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(85)
features = ["signal", "aux"]
X = rng.normal(size=(120, 2))
y = (X[:, 0] + 0.4 * X[:, 1] > 0).astype(int)
X[3, 1] = np.nan
train_rows, val_rows = train_test_split(
    np.arange(len(y)), test_size=0.25, random_state=85, stratify=y)
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=1.0, max_iter=300, random_state=85))
model.fit(X[train_rows], y[train_rows])
reference = model.predict_proba(X[val_rows])[:, 1]
assert model.classes_.tolist() == [0, 1]
validation = pd.DataFrame(X[val_rows], columns=features)
validation.insert(0, "id", [f"{row:03d}" for row in val_rows])

temporary = TemporaryDirectory(prefix="ai85-")
folder = Path(temporary.name)
(folder / "model.pkl").write_bytes(pickle.dumps(model, protocol=5))
(folder / "config.json").write_text(json.dumps({"features": features, "class": 1}), encoding="utf-8")
del model
print("train/validation:", len(train_rows), len(val_rows))
print("artifact có model và config:", all((folder / name).is_file()
                                         for name in ["model.pkl", "config.json"]))

train/validation: 90 30
artifact có model và config: True


File model chứa cả các bước tiền xử lý lẫn model, vì cả ba nằm trong cùng một Pipeline. File config ghi thứ tự các cột đặc trưng và lớp cần lấy xác suất. Biến `model` bị xóa ngay sau khi lưu, nên phần sau chỉ có thể dùng model nạp lại từ file.

Danh sách mẫu validation được giữ trong `val_rows`. Chỉ nhớ seed là không đủ, vì nếu thứ tự dữ liệu đổi, cùng một seed sẽ cho ra một cách chia khác.

In [ ]:
from hashlib import sha256
from sklearn.metrics import log_loss

def infer_from_artifact(directory, frame):
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    loaded = pickle.loads((directory / "model.pkl").read_bytes())
    column = loaded.classes_.tolist().index(config["class"])
    values = frame[config["features"]].to_numpy()
    return loaded.predict_proba(values)[:, column]

try:
    # Cố tình đảo thứ tự cột đầu vào. Config phải đưa các cột về đúng thứ tự lúc train.
    probabilities = infer_from_artifact(folder, validation[["id", "aux", "signal"]])
    assert np.allclose(probabilities, reference, rtol=0, atol=1e-12)
    output = pd.DataFrame({"id": validation["id"], "probability": probabilities})
    destination = folder / "submission.csv"
    output.to_csv(destination, index=False, float_format="%.8f", lineterminator="\n")
    readback = pd.read_csv(destination, dtype={"id": str})
    assert readback.columns.tolist() == ["id", "probability"]
    assert readback["id"].tolist() == validation["id"].tolist()
    assert readback["id"].is_unique
    p = readback["probability"].to_numpy()
    assert np.isfinite(p).all() and ((0 <= p) & (p <= 1)).all()
    scored = log_loss(y[val_rows], np.column_stack([1 - p, p]), labels=[0, 1])
    memory_score = log_loss(y[val_rows], np.column_stack([1 - reference, reference]), labels=[0, 1])
    assert abs(scored - memory_score) < 1e-6
    print("nạp lại trùng dự đoán:", True)
    print("ba hàng đầu:")
    print(readback.head(3).to_csv(index=False, float_format="%.8f", lineterminator="\n").rstrip())
    print("scorer đọc file, log-loss:", f"{scored:.6f}")
    print("sha256 CSV:", sha256(destination.read_bytes()).hexdigest())
finally:
    temporary.cleanup()

nạp lại trùng dự đoán: True
ba hàng đầu:
id,probability
007,0.04982465
025,0.00972475
095,0.98909870
scorer đọc file, log-loss: 0.200701
sha256 CSV: 1020cd91e74c661703d0ebad9c990b7178b66a92f62e08c9ec4889c52e03f11e


Hàm dự đoán chỉ nhận hai thứ: thư mục chứa model, và bảng đặc trưng. Nó không nhận nhãn. Bảng đầu vào bị đảo thứ tự hai cột, nhưng config chọn lại đúng thứ tự, nên xác suất khớp hoàn toàn với model gốc. Nếu không có config, model sẽ lặng lẽ nhận `aux` vào chỗ của `signal`.

Bộ chấm đọc chính file CSV đã ghi ra đĩa, và cho cùng log-loss như khi tính trong bộ nhớ, sai khác dưới một phần triệu. Mã băm SHA-256 xác định đúng nội dung file đã được chấm. Nó chỉ chứng minh file nào đã được chấm, chứ không chứng minh model tốt trên dữ liệu mới. Ví dụ này kiểm việc lưu, nạp lại và chạy lại trong cùng một môi trường. Nó chưa giả lập một máy chấm dùng phiên bản thư viện khác, hay cả một kỳ thi đầy đủ.